# Phase 7 — Evaluation

Computes CLIP alignment, LPIPS diversity, style consistency, and manual ratings across sweep outputs. Run after a full sweep has completed in Colab.

In [ ]:
# Cell 1 — Setup (mount Drive, clone/pull repo, PYTHONPATH)
import os, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/roomify")
if not REPO_DIR.exists():
    !git clone https://github.com/ben-blake/roomify.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

OUTPUTS_DIR = Path("/content/drive/MyDrive/roomify/outputs")

In [ ]:
# Cell 2 — Pick sweep to evaluate
import ipywidgets as widgets
from IPython.display import display

sweeps = sorted(OUTPUTS_DIR.iterdir()) if OUTPUTS_DIR.exists() else []
opts = [s.name for s in sweeps if s.is_dir()]

w = widgets.Dropdown(options=opts, description="Sweep:")
display(w)

In [ ]:
# Cell 3 — CLIP alignment
from roomify.evaluation import clipAlignment

SWEEP = OUTPUTS_DIR / w.value
df_clip = clipAlignment(SWEEP)
print(f"Mean CLIP score: {df_clip['clipScore'].mean():.4f}")
df_clip.sort_values("clipScore", ascending=False).head(10)

In [ ]:
# Cell 4 — LPIPS diversity + style consistency
from roomify.evaluation import lpipsDiversity, styleConsistency

print(f"LPIPS diversity:   {lpipsDiversity(SWEEP):.4f}")
print(f"Style consistency: {styleConsistency(SWEEP):.4f}")

In [ ]:
# Cell 5 — Contact sheet
from roomify.reporting import contactSheet

sheet = contactSheet(SWEEP, thumbSize=256)
sheet.save(str(SWEEP / "contact_sheet.png"))
sheet

In [ ]:
# Cell 6 — Metrics markdown table
from roomify.reporting import metricsTable

from IPython.display import Markdown
Markdown(metricsTable(SWEEP))

In [ ]:
# Cell 7 — Controlled vs uncontrolled comparison
# Requires at least one controlled and one uncontrolled run for the same spec + seed.
import json
from PIL import Image
import matplotlib.pyplot as plt

pairs = []
runs = {}
for rj in sorted(SWEEP.rglob("run.json")):
    d = json.loads(rj.read_text())
    key = (d.get("spec", {}).get("id"), d.get("seed"))
    runs.setdefault(key, {})["ctrl" if d.get("controlled") else "unctrl"] = rj.parent

for key, v in runs.items():
    if "ctrl" in v and "unctrl" in v:
        pairs.append((key, v["unctrl"], v["ctrl"]))

for (spec_id, seed), unctrl, ctrl in pairs[:3]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(Image.open(unctrl / "img_0.png")); axes[0].set_title("Uncontrolled"); axes[0].axis("off")
    axes[1].imshow(Image.open(ctrl / "img_0.png")); axes[1].set_title("Controlled (depth)"); axes[1].axis("off")
    fig.suptitle(f"{spec_id} seed={seed}")
    plt.tight_layout()
    plt.savefig(str(SWEEP / f"comparison_{spec_id}_s{seed}.png"), dpi=150)
    plt.show()

In [ ]:
# Cell 8 — Export top-N images by CLIP score (slide-ready PNGs)
EXPORT_DIR = SWEEP / "top_exports"
EXPORT_DIR.mkdir(exist_ok=True)

top = df_clip.nlargest(6, "clipScore")
for _, row in top.iterrows():
    src = Path(row["imagePath"])
    if src.exists():
        dst = EXPORT_DIR / src.name
        Image.open(src).save(str(dst))
        print(f"Exported: {dst.name}  CLIP={row['clipScore']:.4f}")